# C01. 분류 vs 회귀 — 무엇이 다른가

> 📌 **이 모듈은 분류 보충 트랙입니다.**  
> 5·6차시 정제 다 끝낸 학생, 또는 분류를 더 깊이 알고 싶은 학생용 자기주도 학습 자료.

## 학습 목표

- 회귀와 분류의 차이를 직관적으로 이해
- 같은 펭귄 데이터로 두 문제 비교
- 왜 회귀 모델로 분류가 어려운지 직접 보기

## 사전 지식

- 4차시 (PyTorch 다변량 회귀, 펭귄) 완료
- PyTorch 학습 루프 작성 가능

---


## 1. 같은 데이터, 다른 질문

펭귄 데이터를 떠올려봅시다. 4차시에서 이런 질문을 했어요:

> **회귀**: "부리 길이, 깊이, 지느러미 길이로 **몸무게를 예측**해보자"

이번엔 같은 데이터로 다른 질문:

> **분류**: "부리 길이, 깊이, 지느러미 길이로 **종을 맞춰보자**"

같은 데이터, 다른 문제. **출력의 종류**가 달라요.

| | 회귀 | 분류 |
|---|---|---|
| 타겟 | 연속 수치 (3500g, 4200g, ...) | 카테고리 (Adelie / Gentoo) |
| 출력 | 실수 | 0/1 또는 확률 |
| 손실 함수 | MSE (제곱 오차) | Cross-entropy (다음 모듈) |


## 2. 펭귄 데이터 — 분류 문제로 보기

이번 모듈에선 **이진 분류** (Adelie vs Gentoo)에 집중합니다. 더 단순하니까.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 시뮬레이션 펭귄 데이터 (실제 통계 기반)
np.random.seed(42)
species_stats = {
    'Adelie':    {'n': 152, 'bill_len': (38.8, 2.7), 'flipper': (190, 6.5), 'mass': (3700, 458)},
    'Gentoo':    {'n': 124, 'bill_len': (47.5, 3.1), 'flipper': (217, 6.5), 'mass': (5076, 504)},
}
rows = []
for sp, s in species_stats.items():
    for _ in range(s['n']):
        size = np.random.normal(0, 1)
        rows.append({
            'species': sp,
            'bill_length_mm': s['bill_len'][0] + s['bill_len'][1] * (size * 0.4 + np.random.normal(0, 0.6)),
            'flipper_length_mm': s['flipper'][0] + s['flipper'][1] * (size * 0.7 + np.random.normal(0, 0.4)),
            'body_mass_g': s['mass'][0] + s['mass'][1] * (size * 0.7 + np.random.normal(0, 0.4)),
        })
penguins = pd.DataFrame(rows)
print(f"총 {len(penguins)}마리")
penguins.head()

# Colab에서는 실제 데이터:
# import seaborn as sns
# penguins = sns.load_dataset('penguins').dropna()
# penguins = penguins[penguins['species'].isin(['Adelie', 'Gentoo'])]


## 3. 두 문제를 시각화로 비교


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 회귀 문제 (4차시) - 지느러미 vs 몸무게
axes[0].scatter(penguins['flipper_length_mm'], penguins['body_mass_g'],
                alpha=0.5, color='steelblue', s=30)
axes[0].set_xlabel('Flipper Length (mm)')
axes[0].set_ylabel('Body Mass (g)')
axes[0].set_title('Regression: flipper → body mass')
axes[0].grid(alpha=0.3)

# 오른쪽: 분류 문제 - 종별로 색깔
colors = {'Adelie': '#FF8C00', 'Gentoo': '#008B8B'}
for sp, c in colors.items():
    sub = penguins[penguins['species'] == sp]
    axes[1].scatter(sub['bill_length_mm'], sub['flipper_length_mm'],
                    alpha=0.6, color=c, s=30, label=sp)
axes[1].set_xlabel('Bill Length (mm)')
axes[1].set_ylabel('Flipper Length (mm)')
axes[1].set_title('Classification: features → species')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


**왼쪽**: y가 연속값. 점들이 직선 추세 → 회귀로 잘 잡힘.  
**오른쪽**: y가 카테고리 (색). 두 그룹이 시각적으로 분리됨 → 분류 문제.


## 4. 회귀 모델로 분류 시도해보자 — 왜 안 될까?

라벨을 숫자로 (Adelie=0, Gentoo=1) 만들고 그냥 회귀 모델을 학습시켜봅시다.


In [ ]:
import torch

# 라벨 수치화
y_labels = (penguins['species'] == 'Gentoo').astype(int).values
print(f"Adelie 개수: {(y_labels == 0).sum()}")
print(f"Gentoo 개수: {(y_labels == 1).sum()}")


In [ ]:
# 입력 정규화
def normalize(s):
    return (s - s.mean()) / s.std()

X = torch.tensor(np.column_stack([
    normalize(penguins['bill_length_mm']).values,
    normalize(penguins['flipper_length_mm']).values,
]), dtype=torch.float32)

y = torch.tensor(y_labels, dtype=torch.float32)
print(f"X: {X.shape}, y: {y.shape}")


In [ ]:
# 4차시와 똑같은 회귀 학습 코드 (분류 시도)
w = torch.zeros(2, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
eta = 0.1
epochs = 500

for epoch in range(epochs):
    y_hat = X @ w + b  # 선형 출력 (분류엔 부적합!)
    loss = ((y - y_hat) ** 2).mean()  # MSE
    loss.backward()
    with torch.no_grad():
        w -= eta * w.grad
        b -= eta * b.grad
        w.grad.zero_()
        b.grad.zero_()

# 예측값 분포 확인
y_pred = (X @ w + b).detach().numpy()
print(f"y_pred 범위: {y_pred.min():.3f} ~ {y_pred.max():.3f}")


In [ ]:
# 예측값 vs 실제 라벨
plt.figure(figsize=(10, 4))
plt.scatter(range(len(y)), y_pred, alpha=0.5, label='Prediction')
plt.scatter(range(len(y)), y_labels, alpha=0.3, marker='x', label='True label (0 or 1)')
plt.axhline(0, color='gray', linewidth=0.5)
plt.axhline(1, color='gray', linewidth=0.5)
plt.axhline(0.5, color='red', linestyle='--', label='Threshold 0.5')
plt.xlabel('Sample index')
plt.ylabel('Output value')
plt.title('Linear regression output for classification — what is wrong?')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**문제점이 두 가지 보이세요?**

1. **출력이 0/1을 벗어남**: 어떤 펭귄은 -0.3, 어떤 펭귄은 1.5 같은 값.  
   → 0/1로 해석하려면 0.5 기준으로 자르긴 하는데... 어색.

2. **"멀리서 맞춘 것"이 페널티를 받음**: 진짜 Gentoo인데 예측값 2.0이면 "너무 자신 있어"라며 MSE는 1.0의 제곱오차로 처벌.  
   → 분류에선 자신 있게 맞췄으면 좋은데, MSE는 그걸 벌함.

이런 두 문제 때문에 **분류 전용 도구**가 필요해요:
- 출력을 0~1 사이로 짜내는 함수 → **시그모이드** (C02)
- 확률 기반 손실 함수 → **Cross-entropy** (C03)


## 5. 다음 모듈 미리보기

- **C02 시그모이드**: 출력을 0~1로 변환하는 함수
- **C03 Cross-entropy**: 확률을 평가하는 손실 함수
- **C04 펭귄 종 분류 코드**: 위 두 가지를 4차시 코드에 결합

다음 모듈로!


## 6. ⚠️ 함정 / 주의사항

### 6.1 라벨 인코딩 일관성
이진 분류는 (0, 1)이 표준. (-1, 1)이나 (1, 2)는 비표준.

### 6.2 클래스 불균형
한 클래스가 다른 클래스보다 훨씬 많으면 (90% vs 10%) 모델이 다수 클래스만 맞춰도 정확도 높음.  
**해결**: 정확도 대신 precision/recall, F1 score 사용.

### 6.3 다중 분류는?
3개 이상 클래스 (예: 3종 펭귄)는 시그모이드 대신 **소프트맥스** + **categorical cross-entropy** 사용.  
이 보충 트랙에서는 이진 분류만 다룹니다.
